# MLP — cfDNA Cancer Classifier

**Why MLP after RF and XGBoost:**  
RF and XGBoost are both tree-based — they partition the feature space with axis-aligned cuts. An MLP (Multi-Layer Perceptron) learns non-linear interactions *across* features by composing weighted sums through activation functions. It can potentially capture relationships that trees miss.

The trade-off: MLPs have many more parameters and are more sensitive to regularisation choices, especially with only 148 training samples.

**Architecture choice:**  
We use `sklearn.neural_network.MLPClassifier` — consistent with the RF/XGBoost workflow (same CV strategy, same train/test split, same metric).

Key regularisation levers:
- `hidden_layer_sizes` — number of neurons per layer. Smaller = less capacity = less overfitting. On 148 samples, 1–2 small layers (16–64 neurons) is the right range.
- `alpha` — L2 weight penalty. Higher = smaller weights = smoother decision boundary.
- `early_stopping` — stops training when validation loss stops improving. Prevents over-training even if `max_iter` is large.
- `learning_rate_init` — step size for gradient updates. Too large = unstable; too small = slow convergence.

**Class imbalance:**  
`MLPClassifier` does not support `class_weight='balanced'` directly. We pass `sample_weight` to the `.fit()` call instead — same effect: cancer samples are upweighted to compensate for 95:53 imbalance.

**Evaluation strategy:**  
Same as RF/XGBoost — 5-fold stratified CV on 148 training samples. Test set still locked.

## Step 1 — Load split data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, pathlib

ROOT      = pathlib.Path().resolve().parent
SPLIT_DIR = ROOT / 'data' / 'processed' / 'split'

X_train   = np.load(SPLIT_DIR / 'X_train.npy')
y_train   = np.load(SPLIT_DIR / 'y_train.npy')
feat_cols = np.load(SPLIT_DIR / 'feat_cols.npy', allow_pickle=True).tolist()

print(f'X_train : {X_train.shape}')
print(f'y_train : {y_train.shape}  (healthy={(y_train==0).sum()}  cancer={(y_train==1).sum()})')
print(f'Features: {len(feat_cols)}')

## Step 2 — Baseline MLP with 5-fold CV

**Architecture — why these defaults:**  
`hidden_layer_sizes=(64, 32)` gives two layers. The first learns combinations of the 18 input features; the second compresses those into more abstract representations before the output. 64→32 is a funnel — starts wide, narrows toward the binary decision.

On 148 samples this is already on the edge of too much capacity. We'll tune it down if needed.

**Why `early_stopping=True`:**  
MLPClassifier will split 10% of training data as an internal validation set and stop when its loss stops improving. This prevents over-training without needing to manually set `max_iter` precisely. `n_iter_no_change=20` gives it 20 epochs of patience before stopping.

**Why `sample_weight` instead of `class_weight`:**  
sklearn's MLPClassifier doesn't accept `class_weight`. We compute equivalent per-sample weights using `compute_sample_weight('balanced', y_train)` — cancer samples get weight ≈1.79, healthy samples get weight ≈1.0. Same effect as RF's `class_weight='balanced'`.

**Note on `cross_val_score` + `sample_weight`:**  
`cross_val_score` doesn't pass `sample_weight` through automatically. We use `cross_validate` with `fit_params` to thread it through correctly — otherwise every fold trains without class balancing.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

sample_weights = compute_sample_weight('balanced', y_train)
print(f'Sample weight — healthy: {sample_weights[y_train==0][0]:.4f}  '
      f'cancer: {sample_weights[y_train==1][0]:.4f}')

def mlp_cv(hidden_layer_sizes, alpha, lr, X, y, cv):
    scores = []
    for train_idx, val_idx in cv.split(X, y):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        sw = compute_sample_weight('balanced', y_tr)

        m = MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            activation='relu',
            solver='adam',
            alpha=alpha,
            learning_rate_init=lr,
            max_iter=500,
            early_stopping=True,
            n_iter_no_change=20,
            validation_fraction=0.1,
            random_state=42,
        )
        m.fit(X_tr, y_tr, sample_weight=sw)
        proba = m.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, proba))
    return np.array(scores)

scores = mlp_cv((64, 32), alpha=1e-4, lr=1e-3, X=X_train, y=y_train, cv=cv)

print()
print('Baseline MLP — 5-fold CV AUC')
print(f'  Fold scores : {[round(s, 4) for s in scores]}')
print(f'  Mean AUC    : {scores.mean():.4f}')
print(f'  Std         : {scores.std():.4f}')
print(f'  Range       : {scores.min():.4f} – {scores.max():.4f}')
print()
print(f'RF (tuned)  : 0.8389')
print(f'XGB (tuned) : 0.8113')
print(f'MLP base    : {scores.mean():.4f}')

## Step 3 — Hyperparameter Tuning

**What we're searching:**

- `hidden_layer_sizes` — architecture of the network. Smaller networks overfit less on 148 samples. We try single-layer (32,) and (64,) and two-layer (32, 16) and (64, 32).
- `alpha` — L2 weight penalty. Controls how strongly large weights are penalised. Higher alpha = smaller weights = smoother decision boundary. Range: 1e-4 to 1e-1.
- `learning_rate_init` — step size for Adam optimiser. Lower = more stable but slower convergence.

**Why grid search manually here:**  
`sklearn.GridSearchCV` doesn't support `sample_weight` via `fit_params` in newer sklearn. Our `mlp_cv` helper already handles the per-fold sample weighting correctly — so we just iterate over the grid ourselves.

In [ ]:
import itertools

architectures = [(32,), (64,), (32, 16), (64, 32)]
alphas        = [1e-4, 1e-3, 1e-2, 1e-1]
lrs           = [1e-3, 5e-4]

results = []
total = len(architectures) * len(alphas) * len(lrs)
done  = 0

for arch, alpha, lr in itertools.product(architectures, alphas, lrs):
    s = mlp_cv(arch, alpha=alpha, lr=lr, X=X_train, y=y_train, cv=cv)
    results.append({'arch': arch, 'alpha': alpha, 'lr': lr,
                    'mean_auc': s.mean(), 'std_auc': s.std()})
    done += 1
    if done % 8 == 0:
        print(f'  {done}/{total} done...')

results_df = (pd.DataFrame(results)
              .sort_values('mean_auc', ascending=False)
              .reset_index(drop=True))

print()
print('Top 10 configurations:')
print(results_df.head(10).to_string(index=False))
print()
best = results_df.iloc[0]
print(f'Best arch  : {best.arch}')
print(f'Best alpha : {best.alpha}')
print(f'Best lr    : {best.lr}')
print(f'Best CV AUC: {best.mean_auc:.4f}  (std {best.std_auc:.4f})')
print(f'Baseline   : 0.8043')

## Step 4 — Final model fit + Save

Refit on all 148 training samples with best params, save to `models/`.

**Why alpha=0.1 as tiebreaker:**  
All (64,) alpha values tied at 0.8214. When hyperparameters tie on AUC, prefer the *more* regularised choice (higher alpha) — it generalises better to unseen data by keeping weights small.

In [ ]:
mlp_final = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    solver='adam',
    alpha=0.1,
    learning_rate_init=1e-3,
    max_iter=500,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.1,
    random_state=42,
)

sw_all = compute_sample_weight('balanced', y_train)
mlp_final.fit(X_train, y_train, sample_weight=sw_all)

with open(ROOT / 'models' / 'mlp_final.pkl', 'wb') as f:
    pickle.dump(mlp_final, f)

print('Saved: models/mlp_final.pkl')
print(f'  Stopped at epoch : {mlp_final.n_iter_}')
print()
print('=' * 50)
print('ALL TABULAR MODELS — CV AUC COMPARISON')
print('=' * 50)
print(f'  Random Forest (tuned) : 0.8389')
print(f'  MLP           (tuned) : 0.8214')
print(f'  XGBoost       (tuned) : 0.8113')
print('=' * 50)
print()
print('Test set is still locked. Final AUC reported in evaluation notebook.')